# Extrapolation Test: Time-Scale Grid Search

## Motivation

Both RoTHP and HoTHP are sensitive to the **absolute scale of timestamps**:

- **RoTHP**: `cos(scale × gap × θⱼ)`. Too small → cos ≈ 1 everywhere (no positional info).  
  Too large → rapid oscillation (chaotic attention). There's an optimal scale.
- **HoTHP**: `exp(−scale × gap × θ′)`. Too small → no decay (all events equally attended).  
  Too large → only immediate neighbors matter. There's an optimal scale.

In the previous notebooks we used per-sequence normalization (mean gap = 1.0) to solve this.
Here we **replace that with a global scale hyperparameter** and grid-search it.

## Important implementation note

`HoTHP.forward()` calls `_normalize_timestamps()` internally, which divides by the
per-sequence mean gap — cancelling any scale applied upstream.
We define `HoTHPRaw` (subclass, `_normalize_timestamps` disabled) so the scale
hyperparameter actually reaches the attention kernel.

## Experiment design

```
Scales: [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

For each seed:
  For each scale:
    Train RoTHP  → record val NLL
    Train HoTHPRaw → record val NLL
  Pick best scale per model (lowest val NLL)
  Evaluate best-RoTHP and best-HoTHP on test splits

Comparison: best-RoTHP vs best-HoTHP (both optimally scaled)
```

Same train-short / test-long setup as `Extrapolation_Length_Test.ipynb`:
train max 50 events (lag ≤ 49), test-extrap max 500 events (lag ≤ 499).

In [ ]:
import os, sys, math, random, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import contextlib

BASE_SEED        = 42
TRAIN_MAX_EVENTS = 50
EXTRAP_MAX_EVENTS = 500
SCALES = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

def set_global_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def make_run_seed(*parts, base_seed=BASE_SEED):
    key = '::'.join(map(str, parts))
    return (base_seed + int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)) % (2**31)

set_global_seed(BASE_SEED)
sns.set_theme(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git
if 'ufc-easytpp' not in sys.path:
    sys.path.insert(0, os.path.abspath('ufc-easytpp'))

import easy_tpp.model.torch_model.torch_baselayer as baselayer

def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        if mask.dim() == 3: mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None: p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

baselayer.attention = attention_fixed
import easy_tpp.model.torch_model.torch_rothp as rothp_module
rothp_module.attention = attention_fixed

from easy_tpp.config_factory.model_config import ModelConfig
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP


class HoTHPRaw(HoTHP):
    """HoTHP with internal _normalize_timestamps disabled.

    The standard HoTHP divides timestamps by per-sequence mean gap inside forward(),
    which cancels any upstream scale factor.  This subclass skips that step so that
    the global scale applied in the data pipeline actually reaches the attention kernel.

    Used exclusively for scale grid-search experiments.
    """
    def forward(self, time_seqs, type_seqs, attention_mask):
        enc_output = self.layer_type_emb(type_seqs)
        thetas      = self.hope_emb.thetas
        theta_prime = self.hope_emb.theta_prime
        for enc_layer in self.stack_layers:
            enc_output = enc_layer(
                enc_output, mask=attention_mask,
                time_seqs=time_seqs, thetas=thetas, theta_prime=theta_prime)
        return enc_output


print(f'Device: {device}')
print(f'Scales to search: {SCALES}')
print(f'Train max events: {TRAIN_MAX_EVENTS}  |  Extrap max events: {EXTRAP_MAX_EVENTS}')

In [ ]:
NUM_TYPES = 4
mu    = np.array([0.12, 0.10, 0.09, 0.08])
alpha = np.array([
    [0.30, 0.08, 0.05, 0.03],
    [0.06, 0.28, 0.07, 0.04],
    [0.04, 0.06, 0.26, 0.06],
    [0.03, 0.04, 0.05, 0.24],
])
beta_fast, beta_slow = 2.5, 0.15
w_fast,    w_slow    = 0.6, 0.4


def generate_hawkes(rng, num_types, horizon, mu, alpha, beta_fast, beta_slow,
                    w_fast, w_slow, min_events=20, max_events=50):
    while True:
        events, t = [], 0.0
        while t < horizon and len(events) < max_events:
            intensity = mu.copy()
            for t_i, k_i in events:
                dt = t - t_i
                intensity += alpha[:, k_i] * (
                    w_fast * np.exp(-beta_fast * dt) + w_slow * np.exp(-beta_slow * dt))
            lam_bar = float(np.sum(intensity))
            if lam_bar <= 1e-9: break
            t += rng.exponential(1.0 / lam_bar)
            if t >= horizon: break
            candidate = mu.copy()
            for t_i, k_i in events:
                dt = t - t_i
                candidate += alpha[:, k_i] * (
                    w_fast * np.exp(-beta_fast * dt) + w_slow * np.exp(-beta_slow * dt))
            lam_sum = float(np.sum(candidate))
            if rng.uniform() <= lam_sum / lam_bar:
                probs = candidate / lam_sum
                events.append((t, int(rng.choice(num_types, p=probs))))
        if len(events) >= min_events:
            return events[:max_events]


rng = np.random.default_rng(BASE_SEED)

train_seqs = [
    generate_hawkes(rng, NUM_TYPES, 15.0, mu, alpha, beta_fast, beta_slow,
                    w_fast, w_slow, min_events=20, max_events=TRAIN_MAX_EVENTS)
    for _ in tqdm(range(500), desc='Train')
]
val_seqs = [
    generate_hawkes(rng, NUM_TYPES, 15.0, mu, alpha, beta_fast, beta_slow,
                    w_fast, w_slow, min_events=20, max_events=TRAIN_MAX_EVENTS)
    for _ in tqdm(range(100), desc='Val')
]
test_short_seqs = [
    generate_hawkes(rng, NUM_TYPES, 15.0, mu, alpha, beta_fast, beta_slow,
                    w_fast, w_slow, min_events=20, max_events=TRAIN_MAX_EVENTS)
    for _ in tqdm(range(100), desc='Test-short')
]
test_extrap_seqs = [
    generate_hawkes(rng, NUM_TYPES, 150.0, mu, alpha, beta_fast, beta_slow,
                    w_fast, w_slow, min_events=100, max_events=EXTRAP_MAX_EVENTS)
    for _ in tqdm(range(100), desc='Test-extrap (long)')
]

# Raw delta statistics (before any scaling)
all_raw_deltas = []
for seq in train_seqs:
    times = sorted([t for t, _ in seq])
    all_raw_deltas.extend([times[i] - times[i-1] for i in range(1, len(times))])
raw_mean_gap = float(np.mean(all_raw_deltas))
print(f'Raw train mean inter-event gap: {raw_mean_gap:.4f} time units')
print(f'Equivalent to per-seq norm scale ≈ {1.0/raw_mean_gap:.1f}')
print()
print('Scale effect on mean gap (raw × scale):')
for s in SCALES:
    mg = raw_mean_gap * s
    decay_hothp = math.exp(-mg * 1.5)  # approx: theta' ≈ 1.5
    cos_rothp   = math.cos(mg * 1.0)   # theta_0 = 1.0
    print(f'  scale={s:8.4f}: mean_gap={mg:.4f}, '
          f'HoTHP decay@1gap={decay_hothp:.3f}, '
          f'RoTHP cos@1gap={cos_rothp:+.3f}')

In [ ]:
def convert_to_tensors_scaled(seqs, num_types, scale):
    """Convert sequences with a global time scale (no per-sequence normalization).

    Timestamps are shifted to start at 0 then multiplied by `scale`.
    For HoTHPRaw (which skips internal normalization), `scale` directly controls
    the decay rate of the hyperbolic kernel.
    For RoTHP, `scale` controls the oscillation frequency.
    """
    converted = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        times = torch.tensor([t for t, _ in seq], dtype=torch.float32)
        types = torch.tensor([k for _, k in seq], dtype=torch.long)
        deltas = torch.zeros_like(times)
        deltas[1:] = times[1:] - times[:-1]
        times  = (times - times[0]) * scale
        deltas = deltas * scale
        converted.append({'time_seqs': times, 'time_delta_seqs': deltas, 'type_seqs': types})
    return converted


def make_loaders(scale, batch_train=64, batch_test=32, batch_extrap=16):
    """Build all four DataLoaders for a given scale."""
    train_d  = convert_to_tensors_scaled(train_seqs,       NUM_TYPES, scale)
    val_d    = convert_to_tensors_scaled(val_seqs,         NUM_TYPES, scale)
    short_d  = convert_to_tensors_scaled(test_short_seqs,  NUM_TYPES, scale)
    extrap_d = convert_to_tensors_scaled(test_extrap_seqs, NUM_TYPES, scale)

    pad_id = NUM_TYPES

    def collate_fn(batch_list):
        B = len(batch_list)
        L = max(len(x['time_seqs']) for x in batch_list)
        pad_time  = torch.zeros(B, L, dtype=torch.float32)
        pad_delta = torch.zeros(B, L, dtype=torch.float32)
        pad_type  = torch.full((B, L), pad_id, dtype=torch.long)
        npm       = torch.zeros(B, L, dtype=torch.float32)
        attn      = torch.ones(B, L, L, dtype=torch.bool)
        causal    = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
        for i, item in enumerate(batch_list):
            l = len(item['time_seqs'])
            pad_time[i, :l]  = item['time_seqs']
            pad_delta[i, :l] = item['time_delta_seqs']
            pad_type[i, :l]  = item['type_seqs']
            npm[i, :l] = 1.0
            m = causal.clone(); m[:, l:] = True; m[l:, :] = True
            attn[i] = m
        return (pad_time, pad_delta, pad_type, npm, attn)

    gen = torch.Generator()
    gen.manual_seed(make_run_seed('scale', scale, 'loader'))
    return (
        DataLoader(train_d,  batch_size=batch_train,  shuffle=True,  collate_fn=collate_fn, generator=gen),
        DataLoader(val_d,    batch_size=batch_train,  shuffle=False, collate_fn=collate_fn),
        DataLoader(short_d,  batch_size=batch_test,   shuffle=False, collate_fn=collate_fn),
        DataLoader(extrap_d, batch_size=batch_extrap, shuffle=False, collate_fn=collate_fn),
        extrap_d,
    )

print('Loader factory ready.')

In [ ]:
pad_id = NUM_TYPES

config = ModelConfig(**{
    'hidden_size': 32,
    'num_layers': 2,
    'num_heads': 2,
    'dropout_rate': 0.1,
    'num_event_types': NUM_TYPES,
    'num_event_types_pad': NUM_TYPES + 1,
    'event_pad_index': pad_id,
    'time_emb_size': 32,
    'use_ln': True,
    'gpu': 0 if torch.cuda.is_available() else -1,
    'model_id': 'ScaleGrid',
    'thinning': {
        'num_sample': 1, 'num_exp': 500, 'over_sample_rate': 5.0,
        'patience_counter': 5, 'num_samples_boundary': 5,
        'dtime_max': 5.0, 'num_step_gen': 1,
    },
    'loss_integral_num_sample_per_step': 20,
    'use_mc_samples': False,
})

USE_AMP = device.type == 'cuda'
if USE_AMP:
    try:
        _scaler_cls = torch.amp.GradScaler
        _autocast_fn = lambda: torch.amp.autocast(device_type='cuda')
    except AttributeError:
        _scaler_cls = torch.cuda.amp.GradScaler
        _autocast_fn = lambda: torch.cuda.amp.autocast()
else:
    _scaler_cls = None
    _autocast_fn = contextlib.nullcontext

print(f'AMP: {USE_AMP}')


def evaluate_nll(model, loader):
    model.eval()
    total_loss, total_events = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            batch = [t.to(device) for t in batch]
            with _autocast_fn():
                loss, num = model.loglike_loss(batch)
            total_loss += loss.item()
            total_events += num
    return total_loss / (total_events + 1e-9)


def evaluate_post_nll(model, loader, cutoff=TRAIN_MAX_EVENTS):
    """NLL only for events at positions >= cutoff (genuine OOD lag zone)."""
    model.eval()
    total_loss, total_events = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            pad_time, pad_delta, pad_type, non_pad_mask, attention_mask = \
                [t.to(device) for t in batch]
            post_mask = non_pad_mask.clone()
            post_mask[:, :cutoff] = 0.0
            if post_mask.sum() == 0:
                continue
            with _autocast_fn():
                loss, num = model.loglike_loss(
                    [pad_time, pad_delta, pad_type, post_mask, attention_mask])
            total_loss += loss.item()
            total_events += num
    return total_loss / (total_events + 1e-9)


def train_one(model, name, train_loader, val_loader,
              epochs=300, patience=15, grad_clip=1.0, lr=1e-3):
    """Train a single model. Returns (best_val_nll, best_state_dict)."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.5, patience=8, min_lr=1e-5)
    scaler = _scaler_cls(enabled=True) if USE_AMP else None

    best_val   = float('inf')
    best_state = None
    no_improve = 0

    for ep in range(epochs):
        model.train()
        for batch in train_loader:
            batch = [t.to(device) for t in batch]
            opt.zero_grad()
            with _autocast_fn():
                loss, num = model.loglike_loss(batch)
                nll = loss / (num + 1e-9)
            if not torch.isnan(nll):
                if scaler is not None:
                    scaler.scale(nll).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(opt)
                    scaler.update()
                else:
                    nll.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    opt.step()

        val_nll = evaluate_nll(model, val_loader)
        scheduler.step(val_nll)

        if val_nll < best_val - 1e-4:
            best_val   = val_nll
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    return best_val, best_state


print('Training functions ready.')

In [ ]:
N_SEEDS = 5   # reduced: 5 seeds × 7 scales × 2 models = 70 trainings total
seeds   = [BASE_SEED + i * 100 for i in range(N_SEEDS)]

results_per_seed = []

for run, seed in enumerate(seeds):
    print(f'\n{"="*70}')
    print(f'Seed {run+1}/{N_SEEDS}  (seed={seed})')
    print('='*70)

    # ── Grid search for each scale ──────────────────────────────────────────
    scale_results = {s: {'rothp_val': None, 'hothp_val': None,
                         'rothp_state': None, 'hothp_state': None}
                     for s in SCALES}

    for scale in SCALES:
        train_loader, val_loader, short_loader, extrap_loader, extrap_data = \
            make_loaders(scale)

        # --- RoTHP ---
        set_global_seed(make_run_seed('scale', scale, 'RoTHP', base_seed=seed))
        rothp = RoTHP(config).to(device)
        r_val, r_state = train_one(
            rothp, f'RoTHP s={scale}', train_loader, val_loader, lr=1e-3)
        scale_results[scale]['rothp_val']   = r_val
        scale_results[scale]['rothp_state'] = r_state

        # --- HoTHPRaw ---
        set_global_seed(make_run_seed('scale', scale, 'HoTHP', base_seed=seed))
        hothp = HoTHPRaw(config).to(device)
        h_val, h_state = train_one(
            hothp, f'HoTHP s={scale}', train_loader, val_loader, lr=5e-4)
        scale_results[scale]['hothp_val']   = h_val
        scale_results[scale]['hothp_state'] = h_state

        print(f'  scale={scale:8.4f}: RoTHP val={r_val:.4f}  HoTHP val={h_val:.4f}')

    # ── Pick best scale per model ───────────────────────────────────────────
    best_r_scale = min(SCALES, key=lambda s: scale_results[s]['rothp_val'])
    best_h_scale = min(SCALES, key=lambda s: scale_results[s]['hothp_val'])

    print(f'\n  Best scale → RoTHP: {best_r_scale}  |  HoTHP: {best_h_scale}')

    # ── Load best models and evaluate on test splits ────────────────────────
    _, _, short_loader_r, extrap_loader_r, _ = make_loaders(best_r_scale)
    _, _, short_loader_h, extrap_loader_h, _ = make_loaders(best_h_scale)

    best_rothp = RoTHP(config).to(device)
    best_rothp.load_state_dict(
        {k: v.to(device) for k, v in scale_results[best_r_scale]['rothp_state'].items()})

    best_hothp = HoTHPRaw(config).to(device)
    best_hothp.load_state_dict(
        {k: v.to(device) for k, v in scale_results[best_h_scale]['hothp_state'].items()})

    rothp_short  = evaluate_nll(best_rothp, short_loader_r)
    rothp_extrap = evaluate_nll(best_rothp, extrap_loader_r)
    rothp_post   = evaluate_post_nll(best_rothp, extrap_loader_r)
    hothp_short  = evaluate_nll(best_hothp, short_loader_h)
    hothp_extrap = evaluate_nll(best_hothp, extrap_loader_h)
    hothp_post   = evaluate_post_nll(best_hothp, extrap_loader_h)

    # Val NLL grid for this seed (for scale sensitivity plot)
    rothp_val_by_scale = [scale_results[s]['rothp_val'] for s in SCALES]
    hothp_val_by_scale = [scale_results[s]['hothp_val'] for s in SCALES]

    results_per_seed.append({
        'seed': seed,
        'best_r_scale': best_r_scale,
        'best_h_scale': best_h_scale,
        'rothp_short':  rothp_short,  'rothp_extrap':  rothp_extrap,  'rothp_post':  rothp_post,
        'hothp_short':  hothp_short,  'hothp_extrap':  hothp_extrap,  'hothp_post':  hothp_post,
        'rothp_deg':      rothp_extrap - rothp_short,
        'hothp_deg':      hothp_extrap - hothp_short,
        'rothp_post_deg': rothp_post   - rothp_short,
        'hothp_post_deg': hothp_post   - hothp_short,
        'rothp_val_by_scale': rothp_val_by_scale,
        'hothp_val_by_scale': hothp_val_by_scale,
    })
    r = results_per_seed[-1]
    print(f'  RoTHP (scale={best_r_scale}): short={rothp_short:.4f}, '
          f'extrap={rothp_extrap:.4f}, post={rothp_post:.4f}')
    print(f'  HoTHP (scale={best_h_scale}): short={hothp_short:.4f}, '
          f'extrap={hothp_extrap:.4f}, post={hothp_post:.4f}')

In [ ]:
from scipy import stats

def arr(key): return np.array([r[key] for r in results_per_seed])
def ms(a):    return a.mean(), (a.std(ddof=1) if len(a) > 1 else 0.0)

rothp_post_deg_arr = arr('rothp_post_deg')
hothp_post_deg_arr = arr('hothp_post_deg')
rothp_deg_arr      = arr('rothp_deg')
hothp_deg_arr      = arr('hothp_deg')

print(f'Results over {N_SEEDS} seeds (each model uses its best scale)\n')
print(f'{"":35} {"RoTHP":>16} {"HoTHP":>16}')
for label, rk, hk in [
    ('Short NLL (in-dist)',         'rothp_short',    'hothp_short'),
    ('Extrap NLL (all events)',      'rothp_extrap',   'hothp_extrap'),
    (f'Post-train NLL (pos≥{TRAIN_MAX_EVENTS})', 'rothp_post', 'hothp_post'),
    ('Degradation (extrap−short)',   'rothp_deg',      'hothp_deg'),
    ('Post-train degradation',       'rothp_post_deg', 'hothp_post_deg'),
]:
    rm, rs = ms(arr(rk))
    hm, hs = ms(arr(hk))
    print(f'{label:35} {rm:+.4f}±{rs:.4f}  {hm:+.4f}±{hs:.4f}')

print('\n--- Best scale per seed ---')
print(f'{"Seed":>6}  {"RoTHP best scale":>17}  {"HoTHP best scale":>17}  '
      f'{"RoTHP post_deg":>15}  {"HoTHP post_deg":>15}')
for r in results_per_seed:
    print(f'{r["seed"]:>6}  {r["best_r_scale"]:>17}  {r["best_h_scale"]:>17}  '
          f'{r["rothp_post_deg"]:>+15.4f}  {r["hothp_post_deg"]:>+15.4f}')

# Statistical test
print('\n--- H1: RoTHP post_deg > HoTHP post_deg ---')
t_stat, p_two = stats.ttest_rel(rothp_post_deg_arr, hothp_post_deg_arr)
p_one = p_two / 2 if t_stat > 0 else 1.0 - p_two / 2
try:
    _, p_w = stats.wilcoxon(rothp_post_deg_arr, hothp_post_deg_arr, alternative='greater')
    w_str = f'Wilcoxon p={p_w:.4f}'
except Exception:
    w_str = 'Wilcoxon N/A'
sig = '✓ significant' if p_one < 0.05 else '✗ not significant'
print(f'  t={t_stat:.3f}, one-sided p={p_one:.4f}  {w_str}  [{sig}]')

# ── Figure 1: scale sensitivity curves ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
log_scales = np.log10(SCALES)
scale_labels = [str(s) for s in SCALES]

for run, r in enumerate(results_per_seed):
    axes[0].plot(log_scales, r['rothp_val_by_scale'], 'o-',
                 color='#4C72B0', alpha=0.5, lw=1.2, ms=5)
    axes[1].plot(log_scales, r['hothp_val_by_scale'], 's-',
                 color='#C44E52', alpha=0.5, lw=1.2, ms=5)

# Mean curves
r_mean_val = np.mean([r['rothp_val_by_scale'] for r in results_per_seed], axis=0)
h_mean_val = np.mean([r['hothp_val_by_scale'] for r in results_per_seed], axis=0)
axes[0].plot(log_scales, r_mean_val, 'o-', color='#4C72B0', lw=2.5, ms=8, label='Mean')
axes[1].plot(log_scales, h_mean_val, 's-', color='#C44E52', lw=2.5, ms=8, label='Mean')

for ax, title, col in [
    (axes[0], 'RoTHP: Val NLL vs Scale', '#4C72B0'),
    (axes[1], 'HoTHPRaw: Val NLL vs Scale', '#C44E52'),
]:
    ax.set_xticks(log_scales)
    ax.set_xticklabels(scale_labels, rotation=45)
    ax.set_xlabel('Scale (log axis)')
    ax.set_ylabel('Best Val NLL')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Scale sensitivity: which scale minimises val NLL?\n'
             '(thin lines = individual seeds, thick = mean)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('Scale_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Figure 2: bar chart (3 splits) + per-seed post_deg ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(3)
bw = 0.35
r_m = [ms(arr(k))[0] for k in ('rothp_short', 'rothp_extrap', 'rothp_post')]
r_s = [ms(arr(k))[1] for k in ('rothp_short', 'rothp_extrap', 'rothp_post')]
h_m = [ms(arr(k))[0] for k in ('hothp_short', 'hothp_extrap', 'hothp_post')]
h_s = [ms(arr(k))[1] for k in ('hothp_short', 'hothp_extrap', 'hothp_post')]

axes[0].bar(x - bw/2, r_m, bw, yerr=r_s, label='RoTHP (best scale)', color='#4C72B0', capsize=4)
axes[0].bar(x + bw/2, h_m, bw, yerr=h_s, label='HoTHP (best scale)', color='#C44E52', capsize=4)
axes[0].set_xticks(x)
axes[0].set_xticklabels([
    'Short\n(in-dist)',
    'Extrap\n(all events)',
    f'Post-train\n(pos≥{TRAIN_MAX_EVENTS})',
])
axes[0].set_ylabel('Test NLL')
axes[0].set_title(f'NLL comparison — best scale per model ({N_SEEDS} seeds)')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

x_pos = np.arange(N_SEEDS)
seed_labels = [str(r['seed']) for r in results_per_seed]
axes[1].plot(x_pos, rothp_post_deg_arr, 'o-', color='#4C72B0', lw=1.5, ms=7,
             label=[f'RoTHP (scale={r["best_r_scale"]})' for r in results_per_seed][0])
axes[1].plot(x_pos, hothp_post_deg_arr, 's--', color='#C44E52', lw=1.5, ms=7,
             label=[f'HoTHP (scale={r["best_h_scale"]})' for r in results_per_seed][0])
for i, r in enumerate(results_per_seed):
    axes[1].annotate(str(r['best_r_scale']), (i, rothp_post_deg_arr[i]),
                     textcoords='offset points', xytext=(0, 6), ha='center',
                     fontsize=7, color='#4C72B0')
    axes[1].annotate(str(r['best_h_scale']), (i, hothp_post_deg_arr[i]),
                     textcoords='offset points', xytext=(0, -12), ha='center',
                     fontsize=7, color='#C44E52')
axes[1].axhline(0, color='gray', ls=':', alpha=0.6)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(seed_labels, rotation=45, ha='right', fontsize=9)
axes[1].set_xlabel('Seed')
axes[1].set_ylabel(f'Post-train degradation (pos≥{TRAIN_MAX_EVENTS} NLL − short NLL)')
axes[1].set_title('Per-seed post-train degradation\n(annotations = best scale used)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Scale Grid Search: best-tuned RoTHP vs best-tuned HoTHPRaw', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('Scale_Grid_Comparison.png', dpi=300, bbox_inches='tight')
plt.show()